# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boda-25/FlyRank-machine-learning/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Method choice and why

**Selected Lane: Lane 2 — Refresh / Content Opportunity Scoring**

My project focuses on identifying content pages that deserve priority review based on their observed performance and content characteristics.

I will use two supervised classification methods:

**1. Logistic Regression**

I chose Logistic Regression as an interpretable starting model. It provides a simple way to understand how different content and performance signals relate to the observed declining label.

**2. Random Forest**

I will also use Random Forest because it can capture non-linear relationships and interactions between features that a linear model may miss.

**Target variable:**

`is_declining_label`

This target represents whether a page has an observed declining trend in the current dataset.

**Evaluation approach:**

I will compare both models against my Week-4 rule-based baseline using Precision@20, Precision@50, Precision@100, Average Precision, and ROC-AUC.

The objective is not to choose the most complex model, but to determine whether a learned model provides better ranking performance than the simple baseline.

The model output will be used for decision support and manual content review, not as proof that a page will decline in the future.


In [2]:
# ==========================================
# 1. IMPORTS AND DATA PREPARATION
# ==========================================

import pandas as pd
import numpy as np
import warnings

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

# Load the actual starter dataset
DATA_URL = (
    "https://raw.githubusercontent.com/Boda-25/"
    "FlyRank-machine-learning/main/data/raw/"
    "content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

# Remove duplicate content records
df = df.drop_duplicates(subset=["content_id"]).copy()

# Create the target from the documented label source
df["target"] = (
    df["trend_direction"]
    .astype("string")
    .str.lower()
    .eq("down")
    .astype(int)
)

# Create log features from observed 90-day totals first
for col in [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ai_sessions_90d"
]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df["log_" + col] = np.log1p(
        df[col].clip(lower=0)
    )

# Numeric features
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

# Categorical features
categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier"
]

# Check required columns (now that log features exist)
required_columns = (
    numeric_features
    + categorical_features
    + ["content_id", "client_id", "trend_direction",
       "impressions_90d", "days_since_last_update"]
)

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

# Convert numeric columns safely
for col in numeric_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# In the dictionary, avg_position = 0 means no data
df.loc[df["avg_position"] == 0, "avg_position"] = np.nan

# Model inputs
X = df[numeric_features + categorical_features].copy()
y = df["target"].copy()
groups = df["client_id"].fillna("unknown").astype(str)

# Leakage and identifier checks
forbidden_features = [
    "trend_direction",
    "trend_pct",
    "target",
    "is_declining_label",
    "client_id",
    "content_id",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

assert not set(X.columns).intersection(forbidden_features)

assert y.nunique() == 2, "Target must contain both classes."

print("DATA PREPARATION COMPLETE")
print("========================")
print("Dataset rows:", len(df))
print("Unique clients:", groups.nunique())
print("Number of features:", X.shape[1])
print("Target distribution:")
print(y.value_counts())
print("Target positive rate:", round(y.mean(), 4))
print("Leakage check: PASSED")

DATA PREPARATION COMPLETE
Dataset rows: 30000
Unique clients: 32
Number of features: 26
Target distribution:
target
1    16262
0    13738
Name: count, dtype: int64
Target positive rate: 0.5421
Leakage check: PASSED



## 2. Split design

I will use a grouped train/test split based on `client_id`.

Approximately 80% of the clients will be used for training and 20% for testing.

This approach is important because pages belonging to the same client may share patterns and characteristics.

Keeping entire clients separate helps evaluate whether the model can generalize to clients it has not seen during training.

I will use a fixed random seed to make the experiment reproducible.

I will also verify that no client appears in both training and testing sets and that both target classes are represented in each split.

The starter dataset is a single snapshot, so this is a client-holdout evaluation, not a future-time prediction experiment.


In [3]:

# ==========================================
# 2. GROUPED TRAIN / TEST SPLIT
# ==========================================

splitter = GroupShuffleSplit(
    n_splits=100,
    test_size=0.20,
    random_state=RANDOM_STATE
)

selected_split = None

# Find a grouped split containing both classes
for train_idx, test_idx in splitter.split(X, y, groups):

    if (
        y.iloc[train_idx].nunique() == 2
        and y.iloc[test_idx].nunique() == 2
    ):
        selected_split = (train_idx, test_idx)
        break

if selected_split is None:
    raise ValueError(
        "Could not find a valid client-grouped split "
        "containing both target classes. Do not silently "
        "replace it with a row-level split."
    )

train_idx, test_idx = selected_split

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

# Verify there is no client overlap
assert train_clients.isdisjoint(test_clients)

print("SPLIT DESIGN")
print("============")
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))

print("\nTraining target rate:", round(y_train.mean(), 4))
print("Testing target rate:", round(y_test.mean(), 4))

print("\nClient overlap:", len(train_clients & test_clients))
print("Train classes:", sorted(y_train.unique()))
print("Test classes:", sorted(y_test.unique()))

print("\nGrouped split verification: PASSED")


SPLIT DESIGN
Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7

Training target rate: 0.5501
Testing target rate: 0.511

Client overlap: 0
Train classes: [np.int64(0), np.int64(1)]
Test classes: [np.int64(0), np.int64(1)]

Grouped split verification: PASSED



## 3. Train + compare vs my baseline

I will train Logistic Regression and Random Forest using the same training data and the same client-grouped split.

I will compare both models against my Week-4 baseline rule.

The Week-4 baseline prioritizes pages that:
- Have not been updated for at least 180 days.
- Have at least 500 impressions in the last 90 days.

Its score is the impressions count for eligible pages and zero for other pages.

I will evaluate all three approaches on exactly the same test rows.

Precision@20, Precision@50, and Precision@100 are especially important because my lane is about ranking pages for human review.

Average Precision and ROC-AUC will provide additional information about ranking quality.

The baseline will be rebuilt using the same rule rather than comparing results from a different split or evaluation sample.


In [4]:

# ==========================================
# 3. TRAIN MODELS AND COMPARE WITH BASELINE
# ==========================================

# Preprocessing for Logistic Regression
numeric_lr = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("scaler", StandardScaler())
])

# Preprocessing for Random Forest
numeric_rf = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    ))
])

categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_lr = ColumnTransformer([
    ("numeric", numeric_lr, numeric_features),
    ("categorical", categorical_preprocessor, categorical_features)
])

preprocessor_rf = ColumnTransformer([
    ("numeric", numeric_rf, numeric_features),
    ("categorical", categorical_preprocessor, categorical_features)
])

# Define models
models = {
    "Logistic Regression": Pipeline([
        ("preprocessing", preprocessor_lr),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=RANDOM_STATE
        ))
    ]),

    "Random Forest": Pipeline([
        ("preprocessing", preprocessor_rf),
        ("model", RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            min_samples_leaf=25,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ])
}

# Rebuild the exact Week-4 baseline rule
baseline_stale = (
    pd.to_numeric(df["days_since_last_update"], errors="coerce")
    >= 180
)

baseline_visible = (
    pd.to_numeric(df["impressions_90d"], errors="coerce")
    >= 500
)

df["baseline_score"] = np.where(
    baseline_stale & baseline_visible,
    pd.to_numeric(df["impressions_90d"], errors="coerce"),
    0
)

baseline_test_scores = (
    df.iloc[test_idx]["baseline_score"].to_numpy()
)

# Precision at K
def precision_at_k(y_true, scores, k):

    y_array = np.asarray(y_true)
    score_array = np.asarray(scores)

    order = np.argsort(
        -score_array,
        kind="mergesort"
    )

    k = min(k, len(y_array))

    return float(y_array[order[:k]].mean())


# Evaluation function
def evaluate_ranking(name, y_true, scores):

    return {
        "Method": name,

        "Precision@20": precision_at_k(
            y_true, scores, 20
        ),

        "Precision@50": precision_at_k(
            y_true, scores, 50
        ),

        "Precision@100": precision_at_k(
            y_true, scores, 100
        ),

        "Average Precision": average_precision_score(
            y_true, scores
        ),

        "ROC-AUC": roc_auc_score(
            y_true, scores
        ),

        "Test Base Rate": float(np.mean(y_true))
    }


# Baseline results
results = [
    evaluate_ranking(
        "Week-4 Baseline",
        y_test,
        baseline_test_scores
    )
]

trained_models = {}
model_probabilities = {}

# Train and evaluate both models
for name, model in models.items():

    print(f"\nTraining {name}...")

    model.fit(X_train, y_train)

    probabilities = model.predict_proba(X_test)[:, 1]

    trained_models[name] = model
    model_probabilities[name] = probabilities

    results.append(
        evaluate_ranking(
            name,
            y_test,
            probabilities
        )
    )

# Final comparison table
comparison = pd.DataFrame(results)

comparison = comparison.sort_values(
    by="Precision@50",
    ascending=False
).reset_index(drop=True)

print("\nMODEL VS BASELINE COMPARISON")
display(
    comparison.style.format({
        "Precision@20": "{:.4f}",
        "Precision@50": "{:.4f}",
        "Precision@100": "{:.4f}",
        "Average Precision": "{:.4f}",
        "ROC-AUC": "{:.4f}",
        "Test Base Rate": "{:.4f}"
    })
)

print("\nTraining and evaluation completed.")



Training Logistic Regression...

Training Random Forest...

MODEL VS BASELINE COMPARISON


,Method,Precision@20,Precision@50,Precision@100,Average Precision,ROC-AUC,Test Base Rate
0,Logistic Regression,0.8500,0.7600,0.7400,0.6103,0.6170,0.5110
1,Week-4 Baseline,0.8000,0.7000,0.6400,0.5110,0.5000,0.5110
2,Random Forest,0.5500,0.5800,0.6200,0.5906,0.6083,0.5110



Training and evaluation completed.



## 4. Errors and interpretation

I will inspect the Random Forest model's errors on the held-out test clients.

I will examine false positives and false negatives to understand where the model makes incorrect classifications.

A false positive means the model identifies a page as declining when its observed label is not declining.

A false negative means the model misses a page that has an observed declining label.

I will also use permutation importance to identify which input features contribute most to the model's ranking performance.

The most important features will be interpreted as associations within this dataset, not as proof of causation.

Some errors may be explained by seasonal changes, differences between clients, missing measurements, or overlapping content signals.

The final model will only be considered useful if its measured ranking performance justifies using it instead of the simpler Week-4 baseline.


In [5]:

# ==========================================
# 4. ERROR ANALYSIS AND INTERPRETATION
# ==========================================

# Use Random Forest for error inspection
rf_model = trained_models["Random Forest"]

rf_probabilities = model_probabilities["Random Forest"]

rf_predictions = (
    rf_probabilities >= 0.5
).astype(int)

# Build test review table
test_review = df.iloc[test_idx][
    [
        "content_id",
        "impressions_90d",
        "days_since_last_update",
        "ctr",
        "avg_position",
        "content_type"
    ]
].copy()

test_review["actual"] = y_test.to_numpy()
test_review["predicted"] = rf_predictions
test_review["probability"] = rf_probabilities

# Identify errors
test_review["error_type"] = np.select(
    [
        (test_review["predicted"] == 1) &
        (test_review["actual"] == 0),

        (test_review["predicted"] == 0) &
        (test_review["actual"] == 1)
    ],
    [
        "False Positive",
        "False Negative"
    ],
    default="Correct"
)

# Find confidently incorrect cases
errors = test_review[
    test_review["error_type"] != "Correct"
].copy()

errors["wrong_confidence"] = np.where(
    errors["error_type"] == "False Positive",
    errors["probability"],
    1 - errors["probability"]
)

errors = errors.sort_values(
    "wrong_confidence",
    ascending=False
)

print("ERROR SUMMARY")
print("=============")
print("False positives:", int(
    (test_review["error_type"] == "False Positive").sum()
))
print("False negatives:", int(
    (test_review["error_type"] == "False Negative").sum()
))

print("\nTHREE MOST CONFIDENTLY WRONG CASES")

if len(errors) > 0:

    display(
        errors.head(3)[
            [
                "content_id",
                "actual",
                "predicted",
                "probability",
                "error_type",
                "impressions_90d",
                "days_since_last_update",
                "ctr",
                "avg_position",
                "content_type"
            ]
        ]
    )

else:
    print("No classification errors were found at the 0.5 threshold.")

# ==========================================
# PERMUTATION FEATURE IMPORTANCE
# ==========================================

# Create a balanced sample from the test set
sample_indices = []

for class_value in [0, 1]:

    class_indices = y_test[
        y_test == class_value
    ].index

    sample_size = min(500, len(class_indices))

    if sample_size > 0:

        sampled = X_test.loc[class_indices].sample(
            n=sample_size,
            random_state=RANDOM_STATE
        )

        sample_indices.extend(sampled.index.tolist())

X_perm = X_test.loc[sample_indices]
y_perm = y_test.loc[sample_indices]

# Calculate permutation importance
perm_result = permutation_importance(
    rf_model,
    X_perm,
    y_perm,
    scoring="average_precision",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "Feature": X_perm.columns,
    "Importance Mean": perm_result.importances_mean,
    "Importance Std": perm_result.importances_std
})

importance_df = importance_df.sort_values(
    "Importance Mean",
    ascending=False
).reset_index(drop=True)

print("\nTOP 10 PERMUTATION FEATURES")
display(importance_df.head(10))

print("\nERROR ANALYSIS COMPLETED.")


ERROR SUMMARY
False positives: 1316
False negatives: 1279

THREE MOST CONFIDENTLY WRONG CASES


,content_id,actual,predicted,probability,error_type,impressions_90d,days_since_last_update,ctr,avg_position,content_type
1864,content_16f38acf0f26,1,0,0.125312,False Negative,2,20,0.0,50.0,keyword article
27271,content_7bc32bc1df59,1,0,0.144357,False Negative,1,92,0.0,NaN,keyword article
22042,content_2ba626fea4d6,0,1,0.853512,False Positive,360,104,0.0,7.2,keyword article



TOP 10 PERMUTATION FEATURES


,Feature,Importance Mean,Importance Std
0,days_with_impressions,0.018713,0.007206
1,avg_position,0.012907,0.001986
2,ctr,0.009062,0.002331
3,position_tier,0.006535,0.001534
4,log_clicks_90d,0.006314,0.001818
5,log_impressions_90d,0.006299,0.004094
6,impression_tier,0.005263,0.001701
7,scroll_rate,0.003085,0.002405
8,days_with_sessions,0.002677,0.002227
9,search_volume,0.001966,0.002069



ERROR ANALYSIS COMPLETED.



## Self-check

- [ ] Method choice is explained and fits my lane.
- [ ] The target variable and feature selection are clearly defined.
- [ ] The train/test split is grouped by client.
- [ ] No client appears in both training and testing sets.
- [ ] Logistic Regression and Random Forest were trained successfully.
- [ ] The Week-4 baseline was evaluated on the exact same test rows.
- [ ] The comparison table includes Precision@20, Precision@50, Precision@100, Average Precision, ROC-AUC, and test base rate.
- [ ] Model errors were inspected and explained.
- [ ] Permutation feature importance was calculated and interpreted.
- [ ] No target-derived columns, identifiers, or forbidden trend-window inputs were used as model features.
- [ ] The notebook runs from top to bottom without errors.
- [ ] No client names, URLs, or private queries are exposed.
- [ ] The notebook was committed under `work/notebooks/w05_model.ipynb`.
- [ ] My repository URL was submitted on the assignment card.
